In [ ]:
import tifffile as tiff
from pathlib import Path
import pandas as pd

Loading the data for preprocessing

In [ ]:
DataPath = Path("..") / "Training data"

train_tabular = pd.read_csv(DataPath / "train_tabular.csv")
print(f"Tabular shape: {train_tabular.shape}")

Deal with missing data

In [ ]:
#Finding missing data
missing_data = train_tabular.isnull().sum()
missing_data = missing_data[missing_data > 0]
print("Columns with missing data:")
print(missing_data)

sentinel_files = set()
viirs_files = set()

for idx, row in train_tabular.iterrows():
    sentinel_files.add(row['sentinel2_tiff_file_name'])
    viirs_files.add(row['viirs_tiff_file_name'])

print(f"Total referenced sentinel files: {len(sentinel_files)}")
print(f"Total referenced viirs files: {len(viirs_files)}")

train_composite_path = DataPath / "train_composite"

for file in train_composite_path.iterdir():
    if file.name in sentinel_files:
        sentinel_files.remove(file.name)
    elif file.name in viirs_files:
        viirs_files.remove(file.name)
    else:
        print(f"Unreferenced file: {file.name}")

print(f"Missing sentinel files: {len(sentinel_files)}")
print(f"Missing viirs files: {len(viirs_files)}")

print("Data integrity check complete.")

In [ ]:
#Print rows with missing data
if not missing_data.empty:
    print("Rows with missing data:")
    #Row will consist of geolocation_name, quarter_label, year, and tropical_cyclone_wind_risk
    rows_with_missing = train_tabular[train_tabular.isnull().any(axis=1)][['geolocation_name', 'quarter_label', 'year', 'tropical_cyclone_wind_risk']]
    print(rows_with_missing)

    #Make a set of unique geolocation names with missing data
    unique_geolocations_with_missing = set(rows_with_missing['geolocation_name'].unique())

    #Other rows with the same geolocation_name
    for geolocation in unique_geolocations_with_missing:
        similar_rows = train_tabular[train_tabular['geolocation_name'] == geolocation][['geolocation_name', 'quarter_label', 'year', 'tropical_cyclone_wind_risk']]
        similar_rows.sort_values(by=['year', 'quarter_label'], inplace=True)
        print(f"Other rows with geolocation_name {geolocation}:")
        print(similar_rows)
        #Get the most common categorical value tropical_cyclone_wind_risk value for these rows
        most_common_value = similar_rows['tropical_cyclone_wind_risk'].mode()[0]
        print(f"Most common tropical_cyclone_wind_risk value for {geolocation}: {most_common_value}")
        #Fill missing tropical_cyclone_wind_risk values with the most common value
        train_tabular.loc[(train_tabular['geolocation_name'] == geolocation) & (train_tabular['tropical_cyclone_wind_risk'].isnull()), 'tropical_cyclone_wind_risk'] = most_common_value

Convert categorical columns to numeric

In [ ]:
numeric_columns = train_tabular.select_dtypes(include=['number']).columns
print(f"Numeric columns: {numeric_columns.tolist()}")
nonnumeric_columns = train_tabular.select_dtypes(exclude=['number']).columns
print(f"Nonnumeric columns before processing: {nonnumeric_columns.tolist()}")

#Convert geolocation_id into numeric format
location_mapping = {loc: idx for idx, loc in enumerate(train_tabular['geolocation_name'].unique())}
train_tabular['geolocation_name'] = train_tabular['geolocation_name'].map(location_mapping).astype('Int64')

#Convert quarter_labels into a numeric format like 2020.25 for 2020-Q1, 2019.5 for 2019-Q2, etc.
quarter_mapping = {}
for row in train_tabular['quarter_label'].unique():
    quarter_mapping[row] = int(row.split('-')[1][1])

train_tabular['quarter_label'] = train_tabular['quarter_label'].map(quarter_mapping)
#print(f"Unique quarter labels: {train_tabular['quarter_label'].nunique()}")
#print("Columns after processing:")
#print(train_tabular['quarter_label'].head())

#Convert yes/no columns to 1/0
yes_no_columns = ['developed_country', 'landlocked', 'access_to_airport', 'access_to_port', 'access_to_highway', 'access_to_railway', 'flood_risk_class']
for col in yes_no_columns:
    train_tabular[col] = train_tabular[col].map({'Yes': 1, 'No': 0}).astype('Int64')
    #print(f"Converted column {col} to numeric.")
    #print(train_tabular[col].head())

#Convert country to "japan" = 0 and "philipines" = 1
train_tabular["country"] = train_tabular["country"].map({'Philippines': 0, 'Japan': 1}).astype('Int64')


#Convert region_economic_classification to numeric codes
economic_map = {
    'Low income': 0,
    'Lower-middle income': 1,
    'Upper-middle income': 2,
    'High income': 3
}
train_tabular['region_economic_classification'] = train_tabular['region_economic_classification'].map(economic_map).astype('Int64')
#print(f"Converted 'region_economic_classification' to numeric codes.")
#print(train_tabular['region_economic_classification'].head())

risk_columns = ['seismic_hazard_zone', 'tropical_cyclone_wind_risk', 'tornadoes_wind_risk']
risk_map = {
    'Very Low': 0,
    'Low': 1,
    'Moderate': 2,
    'High': 3,
    'Very High': 4
}
for col in risk_columns:
    train_tabular[col] = train_tabular[col].map(risk_map).astype('Int64')
    #print(f"Converted column {col} to numeric.")
    #print(train_tabular[col].head())

train_tabular['koppen_climate_zone'] = train_tabular['koppen_climate_zone'].astype('category').cat.codes.astype('Int64')

nonnumeric_columns = train_tabular.select_dtypes(exclude=['number']).columns
categorical_columns = train_tabular.columns.difference(numeric_columns).difference(nonnumeric_columns)
print(f"Nonnumeric columns: {nonnumeric_columns.tolist()}")
print(f"Categorical columns: {categorical_columns.tolist()}")

print(train_tabular.columns)

Since the data is split into two countries Japan and Philippines, the data is split to train two separate models. For each country to enhance model performance.

In [ ]:
# Split the data based on country
philipines = pd.DataFrame()
japan = pd.DataFrame()

for row in train_tabular.itertuples(index=False):
    if row.country == "Japan" or row.country == 1:
        japan = pd.concat([japan, pd.DataFrame(data=[row])], ignore_index=True)
    elif row.country == "Philippines" or row.country == 0:
        philipines = pd.concat([philipines, pd.DataFrame(data=[row])], ignore_index=True)
    else:
        raise ValueError(f"Unknown country: {row.country}")

print(f"All data shape: {train_tabular.shape}")
print(f"Philippines shape: {philipines.shape}")
print(f"Japan shape: {japan.shape}")

Normalizing data

In [ ]:
def normalize(df : pd.DataFrame, normalizing_cols  : list) -> pd.DataFrame:

    for col in normalizing_cols:
        min_val = df[col].min()
        max_val = df[col].max()
        if max_val - min_val > 0:
            df[col] = (df[col] - min_val) / (max_val - min_val)
        else:
            df[col] = 0.0
    return df

"""Normalizing the data"""
normalizing_cols = ['year', 'deflated_gdp_usd', 'us_cpi', 'straight_distance_to_capital_km']
philipines = normalize(philipines, normalizing_cols)
japan = normalize(japan, normalizing_cols)
train_tabular = normalize(train_tabular, normalizing_cols)


Saving the data after preprocessing to use for model training.

In [ ]:
SavePath = Path("..") / "Processed data"

#Drop colums with only one unique value
def drop_constant_columns(df):
    for col in df.columns:
        if df[col].nunique() == 1:
            df = df.drop(columns=[col])
    return df

def drop_redundant_columns(df):
    redundant = ['data_id']  #TODO: Add other redundant column names
    for col in redundant:
        if col in df.columns:
            df = df.drop(columns=[col])
    return df

def remove_matching_columns(df):
    exiting_cols = []
    for col in df:
        if col in exiting_cols:
            continue
        for other_col in df:
            if col == other_col:
                continue
            if (df[col] == df[other_col]).all():
                print(f"Column: {col}, matches column: {other_col}")
                exiting_cols.append(other_col)
    df = df.drop(columns=exiting_cols)
    return df

def process(df):
    df = drop_constant_columns(df)
    df = drop_redundant_columns(df)
    df = remove_matching_columns(df)
    return df

train_tabular = process(train_tabular)
philipines = process(philipines)
japan = process(japan)

print(f"All data shape: {train_tabular.shape}")
print(f"Philippines shape: {philipines.shape}")
print(f"Japan shape: {japan.shape}")

train_tabular.to_csv(SavePath / "processed_data.csv", index=False)
philipines.to_csv(SavePath / "processed_philippines.csv", index=False)
japan.to_csv(SavePath / "processed_japan.csv", index=False)